# 第 01–02 章 · 红杯仿真与机械臂动作

选择 **CPU**（硬件加速器选「无」），然后点「运行全部」。两章共用下面这一个安装格。第 01 章先体验手写动作，第 02 章再比较关节空间和任务空间。两个实验都不用下载演示数据。


In [ ]:
# @title 准备仿真环境（两章共用）
import os
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

work = Path('/content/vla-chapter01')
work.mkdir(exist_ok=True)
repo = work / 'VLA-Step-by-Step'
libero = work / 'LIBERO'
libero_revision = '8f1084e3132a39270c3a13ebe37270a43ece2a01'
source_zip = None  # 未推送时可改成上传到 Colab 的源码 ZIP 路径。

def run(*args, **kwargs):
    subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if source_zip is not None:
    repo = work / 'VLA-Step-by-Step-zip'
    with ZipFile(source_zip) as archive:
        archive.extractall(repo)
elif not repo.exists():
    run('git', 'clone', '--depth', '1', 'https://github.com/Blue0w/VLA-Step-by-Step.git', repo)
else:
    # 同一个 Colab 运行时会保留上次下载的仓库，重新准备时先同步实验代码。
    run('git', '-C', repo, 'pull', '--ff-only')
lab = repo / 'chapters/01_environment/lab'
requirements = lab / 'requirements.txt'
if not requirements.is_file():
    raise RuntimeError('没有找到实验代码。请从已推送的分支打开，或上传仓库源码 ZIP。')

if not (libero / 'libero/libero/__init__.py').is_file():
    run('git', 'init', libero)
    run('git', '-C', libero, 'fetch', '--depth', '1',
        'https://github.com/Lifelong-Robot-Learning/LIBERO.git', libero_revision)
    run('git', '-C', libero, 'checkout', '--detach', 'FETCH_HEAD')

run('apt-get', '-qq', 'update')
run('apt-get', '-qq', 'install', '-y', 'xvfb', 'xauth', 'libgl1',
    'libglx-mesa0', 'libgl1-mesa-dri')
run(sys.executable, '-m', 'pip', '-q', 'install', 'uv')
uv = [sys.executable, '-m', 'uv']
venv = work / '.venv'
python = venv / 'bin/python'
if not python.exists():
    run(*uv, 'venv', '--python', '3.12', venv)
run(*uv, 'pip', 'install', '--python', python, '-r', requirements, '-e', repo)

# Colab 用虚拟显示器和 CPU 渲染。两章都通过这里调用本地实验脚本。
env = os.environ.copy()
env.update(PYTHONPATH=str(libero), MUJOCO_GL='glx', PYOPENGL_PLATFORM='glx',
           LIBGL_ALWAYS_SOFTWARE='1', MPLBACKEND='Agg',
           LIBERO_CONFIG_PATH=str(work / 'libero-config'))

def run_simulation(script, *args):
    result = subprocess.run(
        ['xvfb-run', '-a', str(python), '-u', '-X', 'faulthandler',
         str(script), *map(str, args)],
        cwd=repo, env=env, input='n\n', text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout, end='')
    if result.returncode != 0:
        raise RuntimeError(f'仿真运行失败（退出码 {result.returncode}）。请查看上方的日志。')


## 第 01 章 · 手动给机械臂发一条动作

每次改一个参数，重跑本节的参数格和运行格。单步移动可能很难从图片上看出来，可以先看日志里的末端位移。


In [ ]:
# @title 动手试一试
# left 和 right 分别对应把红杯放到左盘和右盘。改这里可以看到任务文字变化。
side = 'left'

# 七个数依次控制 X/Y/Z 方向的移动、末端朝向的变化和夹爪开合。
# 先只改第一项。试 0.8 让 X 方向的移动更明显，试 -0.25 改变方向。
# 每项取值在 [-1, 1]。最后一项 -1 请求张开夹爪，+1 请求合拢。
action = [0.25, 0, 0, 0, 0, 0, -1]

# 每次运行从初始场景开始。改成 5，用同一条动作连续控制五步。
steps = 1

# 相机图像的宽和高，单位是像素。改成 128，比较画面和日志里的图像尺寸。
image_size = 256


In [ ]:
# @title 运行仿真并查看结果
import matplotlib.pyplot as plt

output = work / 'outputs/chapter01'
run_simulation(
    lab / 'demo.py', '--output-dir', output,
    '--side', side, '--steps', steps, '--image-size', image_size,
    '--action', *action,
)

# 仿真成功后，并排显示动作前后的相机画面。
_, axes = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')
for axis, label in zip(axes, ('before', 'after')):
    axis.imshow(plt.imread(output / f'{label}.png'))
    axis.set_title(label)
    axis.set_axis_off()
plt.show()


## 第 02 章 · 对照关节空间和任务空间

两组试验从同一个仿真状态开始。一组只指定一个关节的转动，另一组只指定末端沿 Z 轴移动。对照日志里 J1–J7 的角度变化（度）和末端的 XYZ 位移（毫米）。修改参数后，重跑本节的两个代码格。


In [ ]:
# @title 动手试一试 · 两种控制方式
# 选一个关节，从 1 到 7。先试 J2，再换成 J1，看末端位移方向怎么变。
joint_number = 2

# 关节的转动指令和末端的 Z 轴指令都是 [-1, 1] 的控制器输入。
# 例如把 z_command 改成负值，观察末端位移的第三项是否变号。
joint_command = 0.8
z_command = 0.8

# 两组都连续发送这么多步。输出会打印实际执行的步数。
comparison_steps = 15


In [ ]:
# @title 运行对照并查看结果
import matplotlib.pyplot as plt

output02 = work / 'outputs/chapter02'
run_simulation(
    repo / 'chapters/02_action/lab/demo.py', '--output-dir', output02,
    '--joint', joint_number, '--joint-command', joint_command,
    '--z-command', z_command, '--steps', comparison_steps,
)

_, axes = plt.subplots(2, 2, figsize=(8, 8), layout='constrained')
for row, controller in zip(axes, ('JOINT_POSITION', 'OSC_POSE')):
    for axis, label in zip(row, ['before', 'after']):
        axis.imshow(plt.imread(output02 / f'{controller}-{label}.png'))
        axis.set_title(f'{controller} · {label}')
        axis.set_axis_off()
plt.show()
